Starting the notebook to visualize and test the logic implemented in bloom_filter.py

In [ ]:
import sys, os, math, random, string, time, hashlib

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import pandas as pd
import numpy as np


sys.path.insert(0, os.getcwd())
from bloom_filter import BloomFilter

print("Imports OK")

1. What is a Bloom filter?
A Bloom filter is a probabilistic data structure backed by a bit array.

insert(item) — hash the item k times, set those k bits to 1
contains(item) — hash the item k times, check those k bits
If any bit is 0 → the item is definitely not in the set
If all bits are 1 → the item is probably in the set (false positive possible)

In [ ]:
# create a filter designed for 200 items at 1% false positive rate
demo_filter = BloomFilter(expected_items=200, false_positive_rate=0.01)
print(demo_filter)
print(f"Bit array size : {demo_filter.bit_array_size} bits")
print(f"Hash functions : {demo_filter.num_hash_functions}")

# insert some fruits
fruits = ["apple", "banana", "cherry", "date", "elderberry",
          "fig", "grape", "honeydew", "kiwi", "lemon"]
for fruit in fruits:
    demo_filter.insert(fruit)

print(f"\nInserted {len(fruits)} fruits.\n")

# all inserted fruits must come back True
print("Querying inserted fruits (all should be True):")
for fruit in fruits:
    result = demo_filter.contains(fruit)
    status = "✓" if result else "✗ ERROR"
    print(f"  {status}  {fruit}")

# words never inserted should mostly come back False
not_fruits = ["mango", "nectarine", "orange", "papaya", "quince"]
print("\nQuerying words NOT inserted (should mostly be False):")
for word in not_fruits:
    print(f"  {word}: {demo_filter.contains(word)}")

    


2. Testing the hash functions

Hash function are distributing the positions uniformly across the bit array.
We will check this for three data types here:

- Common English words
- Random strings
- DNA sequences (only characters A, C, G, T)

We will aslo compare the actual fill fraction to the theoretical expectation: 1 - e^(-k·n/m).

In [ ]:
def check_hash_distribution(items, n_expected, fpr=0.01, label=""):
    bf = BloomFilter(expected_items=n_expected, false_positive_rate=fpr)
    for item in items:
        bf.insert(item)

    bits_set = bf.count_bits_set()
    fill = bits_set / bf.bit_array_size

    k, n, m = bf.num_hash_functions, len(items), bf.bit_array_size
    expected_fill = 1 - math.exp(-k * n / m)

    print(f"{label}")
    print(f"  Items inserted : {n}")
    print(f"  Fill fraction  : {fill:.4f}  (expected {expected_fill:.4f})")
    print(f"  Difference     : {abs(fill - expected_fill):.4f}")
    print()


random.seed(0)

# --- English words ---
english_words = [
    "the", "be", "to", "of", "and", "a", "in", "that", "have", "it",
    "for", "not", "on", "with", "he", "as", "you", "do", "at", "this",
    "but", "his", "by", "from", "they", "we", "say", "her", "she", "or",
    "an", "will", "my", "one", "all", "would", "there", "their", "what",
    "so", "up", "out", "if", "about", "who", "get", "which", "go", "me"
]
check_hash_distribution(english_words, n_expected=50, label="English words (50 common words)")

# --- random strings ---
random_strings = ["".join(random.choices(string.ascii_lowercase, k=8)) for _ in range(50)]
check_hash_distribution(random_strings, n_expected=50, label="Random strings (50 × 8 chars)")

# --- DNA sequences ---
dna_sequences = ["".join(random.choices("ACGT", k=20)) for _ in range(50)]
check_hash_distribution(dna_sequences, n_expected=50, label="DNA sequences (50 × 20 bases)")

# --- uniqueness of positions ---
test_bf = BloomFilter(expected_items=1000, false_positive_rate=0.01)
positions = test_bf._get_hash_positions("hello")
print(f"Positions for 'hello' with k={test_bf.num_hash_functions}: {positions}")
print(f"All unique? {len(positions) == len(set(positions))}")

3. False positive rate as the filter fills up
We fill a filter designed for 1000 items at 1% FPR step by step
and measure the actual false positive rate after each batch.
We also fill it beyond its designed capacity to see what happens.

In [ ]:
def measure_fpr(bloom, num_test=2000):
    """Count what fraction of fresh random words the filter wrongly says are present."""
    test_words = ["zz_" + "".join(random.choices(string.ascii_lowercase, k=12)) for _ in range(num_test)]
    hits = sum(1 for w in test_words if bloom.contains(w))
    return hits / num_test


random.seed(7)

designed_n = 1000
target_fpr = 0.01
total_insert = designed_n * 3      # 3× over capacity
batch = total_insert // 30

bf = BloomFilter(expected_items=designed_n, false_positive_rate=target_fpr)
word_pool = ["".join(random.choices(string.ascii_lowercase, k=9)) for _ in range(total_insert)]

counts, actual_rates, theory_rates = [], [], []

for step in range(30):
    for word in word_pool[step * batch: (step + 1) * batch]:
        bf.insert(word)

    actual  = measure_fpr(bf)
    theory  = bf.get_false_positive_rate()

    counts.append(bf.num_inserted)
    actual_rates.append(actual)
    theory_rates.append(theory)


# --- plot ---
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(counts, actual_rates, "o-", color="#e05c5c",label="Actual FPR", linewidth=2, markersize=5)
ax.plot(counts, theory_rates, "--", color="#5c7ae0",label="Theoretical FPR", linewidth=2)
ax.axvline(x=designed_n, color="gray", linestyle=":", linewidth=1.5)
ax.text(designed_n + 30, max(actual_rates) * 0.85,"Designed\ncapacity", fontsize=9, color="gray")
ax.set_xlabel("Items inserted")
ax.set_ylabel("False positive rate")
ax.set_title("False positive rate as Bloom filter fills up")
ax.yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1))
ax.legend()
plt.tight_layout()
plt.savefig("false_positive_rate.png", dpi=150)
plt.show()
print("Saved: false_positive_rate.png")

4. Performance timing (local)

We time insert() and contains() for an increasing numbers of words.
Both operations are O(k) where k is a small constant, so we expect O(1) behaviour.

In [ ]:
sizes = [100, 500, 1000, 5000, 10000, 30000]
insert_us = []   # microseconds per operation
search_us = []

for n in sizes:
    words = ["".join(random.choices(string.ascii_lowercase, k=8)) for _ in range(n)]
    bf = BloomFilter(expected_items=n, false_positive_rate=0.01)

    t0 = time.perf_counter()
    for w in words: 
        bf.insert(w)
    t1 = time.perf_counter()
    insert_us.append((t1 - t0) / n * 1e6)

    t2 = time.perf_counter()
    for w in words: 
        bf.contains(w)
    t3 = time.perf_counter()
    search_us.append((t3 - t2) / n * 1e6)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

In [ ]:
axes[0].plot(sizes, insert_us, "s-", color="#3aaa6e", linewidth=2)
axes[0].set_xlabel("Number of items")
axes[0].set_ylabel("µs per insert")
axes[0].set_title("Average insert time (local)")
axes[0].set_xscale("log")

axes[1].plot(sizes, search_us, "D-", color="#e09a3a", linewidth=2)
axes[1].set_xlabel("Number of items")
axes[1].set_ylabel("µs per lookup")
axes[1].set_title("Average search time (local)")
axes[1].set_xscale("log")

plt.tight_layout()
plt.savefig("timing_local.png", dpi=150)
plt.show()
print("Saved: timing_local.png")

5. HPC benchmark results

The cell below loads the CSV files produced by benchmark.py on the HPC cluster.

In [ ]:
timing_csv = "benchmark_results/timing_results.csv"

if os.path.exists(timing_csv):
    df_time = pd.read_csv(timing_csv)

    fig, axes = plt.subplots(1, 2, figsize=(13, 5))

    axes[0].plot(df_time["num_words"], df_time["total_insert_sec"],
                 "o-", color="#3aaa6e", linewidth=2)
    axes[0].set_xlabel("Number of words")
    axes[0].set_ylabel("Total insert time (s)")
    axes[0].set_title("Total insert time (HPC)")

    axes[1].plot(df_time["num_words"], df_time["total_search_sec"],
                 "D-", color="#e09a3a", linewidth=2)
    axes[1].set_xlabel("Number of words")
    axes[1].set_ylabel("Total search time (s)")
    axes[1].set_title("Total search time (HPC)")

    plt.tight_layout()
    plt.savefig("timing_hpc.png", dpi=150)
    plt.show()
    print("Saved: timing_hpc.png")
    display(df_time)
else:
    print(f"'{timing_csv}' not found.  Run benchmark.py on HPC or locally first.")



6. Compression ratio

How much space does a Bloom filter save compared to storing all items
in a plain Python set?
We compare the logical size (m/8 bytes) for different expected sizes and FPRs.

In [ ]:
comp_csv = "benchmark_results/compression_results.csv"

if os.path.exists(comp_csv):
    df_comp = pd.read_csv(comp_csv)
else:
    # build it on the fly
    bytes_per_set_item = 57
    rows = []
    for n in [100, 500, 1000, 5000, 10000, 50000, 100000]:
        for fpr in [0.01, 0.05, 0.10]:
            bf = BloomFilter(expected_items=n, false_positive_rate=fpr)
            bloom_b = bf.get_memory_bytes()
            set_b   = n * bytes_per_set_item
            rows.append({
                "expected_items": n,
                "false_positive_rate": fpr,
                "bloom_bytes": bloom_b,
                "set_bytes": set_b,
                "compression_ratio": round(set_b / bloom_b, 2),
            })
    df_comp = pd.DataFrame(rows)

colors = ["#5c7ae0", "#e05c5c", "#3aaa6e"]

fig, ax = plt.subplots(figsize=(9, 5))
for i, fpr_val in enumerate(sorted(df_comp["false_positive_rate"].unique())):
    sub = df_comp[df_comp["false_positive_rate"] == fpr_val]
    ax.plot(sub["expected_items"], sub["compression_ratio"],
            "o-", label=f"FPR = {int(fpr_val*100)}%",
            color=colors[i], linewidth=2)

ax.set_xlabel("Expected number of items")
ax.set_ylabel("Compression ratio  (set bytes / bloom bytes)")
ax.set_title("Space saved by using a Bloom filter instead of a plain set")
ax.set_xscale("log")
ax.legend()
plt.tight_layout()
plt.savefig("compression_ratio.png", dpi=150)
plt.show()
print("Saved: compression_ratio.png")